# Hackathon AI & ML — AutoGluon
**Stratégie** : laisser AutoGluon explorer automatiquement LightGBM, XGBoost, CatBoost, Random Forest, ExtraTree, KNN, et leur stacking multi-niveaux.  
**Métrique cible** : F1-score (classe 1).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from autogluon.tabular import TabularPredictor
import warnings
warnings.filterwarnings('ignore')

DATA_DIR      = Path('data')
GROUP_ID      = 'G1'
SUBMISSION_ID = '3'
TIME_LIMIT    = 3600  # secondes — augmenter si possible (ex: 7200 pour 2h)

print('AutoGluon prêt')

## 1. Chargement et préparation

In [ ]:
data     = pd.read_csv(DATA_DIR / 'data.csv')
labels   = pd.read_csv(DATA_DIR / 'ground_truth_train.csv')
test_idx = pd.read_csv(DATA_DIR / 'test_indexes.csv', header=None, names=['SEQN'])
metadata = pd.read_csv(DATA_DIR / 'features_metadata.csv')

# Split train / test
test_seqn  = set(test_idx['SEQN'].values)
train_seqn = set(labels['SEQN'].values) - test_seqn

train_data = data[data['SEQN'].isin(train_seqn)].copy()
test_data  = data[data['SEQN'].isin(test_seqn)].copy()
train_df   = train_data.merge(labels[labels['SEQN'].isin(train_seqn)], on='SEQN')

print(f'Train : {train_df.shape}  |  Test : {test_data.shape}')
print(f'Taux de mortalité : {train_df["MORTSTAT_2019"].mean():.2%}')

In [ ]:
# Features NaN par composante (même feature engineering que advanced.ipynb)
meta_cols  = metadata[['SAS', 'Component']].rename(columns={'SAS': 'col'})
labo_cols  = meta_cols[meta_cols['Component'] == 'Laboratory']['col'].tolist()
exam_cols  = meta_cols[meta_cols['Component'] == 'Examination']['col'].tolist()
quest_cols = meta_cols[meta_cols['Component'] == 'Questionnaire']['col'].tolist()

def add_missing_features(df, labo_cols, exam_cols, quest_cols):
    all_cols = [c for c in df.columns if c not in ['SEQN', 'MORTSTAT_2019']]
    df = df.copy()
    df['feat_nb_missing_total'] = df[all_cols].isnull().sum(axis=1)
    df['feat_pct_missing_total'] = df['feat_nb_missing_total'] / len(all_cols)
    labo_p  = [c for c in labo_cols  if c in df.columns]
    exam_p  = [c for c in exam_cols  if c in df.columns]
    quest_p = [c for c in quest_cols if c in df.columns]
    if labo_p:  df['feat_nb_missing_labo']  = df[labo_p].isnull().sum(axis=1)
    if exam_p:  df['feat_nb_missing_exam']  = df[exam_p].isnull().sum(axis=1)
    if quest_p: df['feat_nb_missing_quest'] = df[quest_p].isnull().sum(axis=1)
    return df

train_df  = add_missing_features(train_df,  labo_cols, exam_cols, quest_cols)
test_data = add_missing_features(test_data, labo_cols, exam_cols, quest_cols)

# Supprimer colonnes >70% manquant
MISSING_THRESHOLD = 0.70
missing_ratio = train_df.isnull().mean()
cols_to_drop  = [c for c in missing_ratio[missing_ratio > MISSING_THRESHOLD].index
                 if c not in ['SEQN', 'MORTSTAT_2019']]

train_clean = train_df.drop(columns=cols_to_drop)
test_clean  = test_data.drop(columns=[c for c in cols_to_drop if c in test_data.columns])

print(f'Colonnes supprimées : {len(cols_to_drop)}')
print(f'Train final : {train_clean.shape}  |  Test final : {test_clean.shape}')

In [ ]:
# Préparer les DataFrames pour AutoGluon
# AutoGluon attend : toutes les features + la colonne cible dans le DataFrame train
# SEQN est un ID — on le retire des features mais on garde pour la soumission

feature_cols    = [c for c in train_clean.columns if c not in ['SEQN', 'MORTSTAT_2019']]
test_seqn_order = test_clean['SEQN']

train_ag = train_clean[feature_cols + ['MORTSTAT_2019']].copy()
test_ag  = test_clean[feature_cols].copy()

# AutoGluon veut la cible en string pour la classification
train_ag['MORTSTAT_2019'] = train_ag['MORTSTAT_2019'].astype(str)

print(f'Train AutoGluon : {train_ag.shape}')
print(f'Test AutoGluon  : {test_ag.shape}')

## 2. Entraînement AutoGluon

`best_quality` : AutoGluon entraîne tous les modèles disponibles + stacking multi-niveaux.  
Ajuster `time_limit` selon le temps disponible (3600 = 1h, 7200 = 2h).

In [ ]:
predictor = TabularPredictor(
    label='MORTSTAT_2019',
    eval_metric='f1',          # optimise directement le F1
    path='autogluon_models',   # sauvegarde les modèles sur disque
    verbosity=2,
)

predictor.fit(
    train_data=train_ag,
    time_limit=TIME_LIMIT,
    presets='best_quality',    # stacking multi-niveaux, tous les modèles
    num_bag_folds=5,           # bagging 5-fold = CV intégrée
    num_stack_levels=2,        # 2 niveaux de stacking
)

## 3. Résultats et comparaison des modèles

In [ ]:
leaderboard = predictor.leaderboard(silent=True)
print(leaderboard[['model', 'score_val', 'pred_time_val', 'fit_time']].to_string())
print(f'\nMeilleur modèle : {predictor.get_model_best()}')
print(f'Meilleur F1 CV  : {leaderboard["score_val"].max():.4f}')
print(f'\nBaseline LGB    : 0.6945')

## 4. Soumission

In [ ]:
y_pred = predictor.predict(test_ag).astype(int)

submission = pd.DataFrame({
    'SEQN': test_seqn_order.values,
    'prediction': y_pred.values
}).sort_values('SEQN')

assert len(submission) == 5000, f'ERREUR : {len(submission)} lignes'

filename = f'{GROUP_ID}_{SUBMISSION_ID}.csv'
submission.to_csv(filename, index=False, header=False)

print(f'Fichier : {filename}')
print(f'Lignes  : {len(submission)}')
print(f'Répartition : {submission["prediction"].value_counts().to_dict()}')

## 5. (Optionnel) Recharger les modèles sans réentraîner

In [ ]:
# Pour recharger sans réentraîner :
# predictor = TabularPredictor.load('autogluon_models')
# y_pred = predictor.predict(test_ag)
print('Décommenter pour recharger les modèles sauvegardés dans autogluon_models/')